In [0]:
%sql

-- STAR SCHEMA
-- One central fact table connected to dimension tables. 
-- not snowflake schema, Because dimensions are not normalized into multiple sub-dimension tables.
USE CATALOG `retail-dwh-project`;
CREATE SCHEMA IF NOT EXISTS gold;
USE SCHEMA gold;


In [0]:

-- DIM CUSTOMER
-- Simple joins → better reporting performance.
CREATE TABLE IF NOT EXISTS gold.dim_customer (
    CustomerSK   BIGINT,
    CustomerID   INT,
    CustomerName STRING,
    Email        STRING,
    City         STRING,
    Address      STRING,
    LastUpdated  DATE,
    StartDate    DATE,
    EndDate      DATE,
    IsActive     INT
);


In [0]:

-- DIM PRODUCT

CREATE TABLE IF NOT EXISTS gold.dim_product (
    ProductSK     BIGINT,
    ProductID     INT,
    ProductName   STRING,
    Category      STRING,
    UnitPrice     DECIMAL(10,2),
    EffectiveDate DATE
);


In [0]:

-- DIM STORE

CREATE TABLE IF NOT EXISTS gold.dim_store (
    StoreSK   BIGINT,
    StoreID   INT,
    StoreName STRING,
    Region    STRING
);


In [0]:

-- FACT SALES
-- Connected to all dimensions.

CREATE TABLE IF NOT EXISTS gold.fact_sales (
    SalesSK       BIGINT,
    TransactionID INT,
    CustomerSK    BIGINT,
    ProductSK     BIGINT,
    StoreSK       BIGINT,
    Quantity      INT,
    Amount        DECIMAL(10,2),
    TxnDate       DATE
);


In [0]:

-- LOAD DIM PRODUCT

INSERT OVERWRITE gold.dim_product
SELECT
    ROW_NUMBER() OVER (ORDER BY ProductID) AS ProductSK,
    ProductID,
    ProductName,
    Category,
    UnitPrice,
    CURRENT_DATE() AS EffectiveDate
FROM clean.products_clean;

-- row_number Generates warehouse surrogate keys CustomerSK and ProductSK

In [0]:

-- LOAD DIM STORE

INSERT OVERWRITE gold.dim_store
SELECT
    ROW_NUMBER() OVER (ORDER BY StoreID) AS StoreSK,
    StoreID,
    StoreName,
    Region
FROM clean.stores_clean;


In [0]:

-- INITIAL LOAD DIM CUSTOMER

-- Loads only new customers

INSERT INTO gold.dim_customer
SELECT
    (SELECT COALESCE(MAX(CustomerSK),0)
     FROM gold.dim_customer)
     + ROW_NUMBER() OVER (ORDER BY CustomerID) AS CustomerSK,

    CustomerID,
    CustomerName,
    Email,
    City,
    Address,
    LastUpdated,

    CURRENT_DATE()     AS StartDate,
    DATE('9999-12-31') AS EndDate,
    1                  AS IsActive

FROM clean.customers_clean

WHERE CustomerID NOT IN (
    SELECT CustomerID
    FROM gold.dim_customer
);


In [0]:

-- SCD TYPE 2 UPDATE logic
-- Old version becomes inactive and then Closes historical record

UPDATE gold.dim_customer
SET
    IsActive = 0,
    EndDate  = CURRENT_DATE()

WHERE IsActive = 1
AND EXISTS (

    SELECT 1
    FROM clean.customers_clean n

   WHERE n.CustomerID   = dim_customer.CustomerID
      AND n.CustomerName = dim_customer.CustomerName
      AND (n.City != dim_customer.City OR n.Address != dim_customer.Address)
);


In [0]:

-- INSERT NEW CUSTOMER VERSION
-- Adds updated customer version with: new city/address, new surrogate key and IsActive = 1
INSERT INTO gold.dim_customer
SELECT
    (SELECT COALESCE(MAX(CustomerSK),0)
     FROM gold.dim_customer)
     + ROW_NUMBER() OVER (ORDER BY n.CustomerID) AS CustomerSK,

    n.CustomerID,
    n.CustomerName,
    n.Email,
    n.City,
    n.Address,
    n.LastUpdated,

    CURRENT_DATE()     AS StartDate,
    DATE('9999-12-31') AS EndDate,
    1                  AS IsActive

FROM clean.customers_clean n

JOIN gold.dim_customer d
ON n.CustomerID = d.CustomerID

WHERE d.IsActive = 0
AND d.EndDate = CURRENT_DATE();


In [0]:

-- LOAD FACT SALES table
-- Joins: customer, product, store and Creates transactional warehouse fact table
INSERT INTO gold.fact_sales
SELECT
    (SELECT COALESCE(MAX(SalesSK),0)
     FROM gold.fact_sales)
     + ROW_NUMBER() OVER (ORDER BY s.TransactionID) AS SalesSK,

    s.TransactionID,
    c.CustomerSK,
    p.ProductSK,
    st.StoreSK,
    s.Quantity,

    ROUND(s.Quantity * p.UnitPrice, 2) AS Amount,

    s.TxnDate

FROM clean.sales_clean s

INNER JOIN gold.dim_customer c
ON s.CustomerID = c.CustomerID
AND c.IsActive = 1

INNER JOIN gold.dim_product p
ON s.ProductID = p.ProductID

INNER JOIN gold.dim_store st
ON s.StoreID = st.StoreID

WHERE s.TransactionID NOT IN (
    SELECT TransactionID
    FROM gold.fact_sales
);


In [0]:

-- VALIDATE GOLD TABLES ROW COUNT

SELECT 'dim_customer' AS table_name, COUNT(*) AS total_rows
FROM gold.dim_customer

UNION ALL

SELECT 'dim_product', COUNT(*)
FROM gold.dim_product

UNION ALL

SELECT 'dim_store', COUNT(*)
FROM gold.dim_store

UNION ALL

SELECT 'fact_sales', COUNT(*)
FROM gold.fact_sales;

GOLD LAYER DATA VALIDATION QUALITY CHECKS

In [0]:
-- FACT TABLE FK VALIDATION
-- Checks null surrogate keys and Ensures all joins successful

SELECT COUNT(*) AS null_sk_rows
FROM gold.fact_sales
WHERE CustomerSK IS NULL
   OR ProductSK  IS NULL
   OR StoreSK    IS NULL;


In [0]:
-- DUPLICATE TRANSACTION CHECK IN FACT
-- Ensures no duplicate sales in fact table.
SELECT
    TransactionID,
    COUNT(*) AS cnt
FROM gold.fact_sales
GROUP BY TransactionID
HAVING COUNT(*) > 1;

In [0]:

-- SCD TYPE 2 VALIDATION
-- Ensures only one current version per customer.

SELECT
    CustomerID,
    COUNT(*) AS active_versions
FROM gold.dim_customer
WHERE IsActive = 1
GROUP BY CustomerID
HAVING COUNT(*) > 1;

In [0]:

-- CHECK CLOSED SCD RECORDS
-- Checks historical customer versions

SELECT *
FROM gold.dim_customer
WHERE IsActive = 0;

In [0]:

-- SALES AMOUNT VALIDATION
-- Ensures no negative/zero revenue
SELECT *
FROM gold.fact_sales
WHERE Amount <= 0;


In [0]:
-- DATE VALIDATION
-- Ensures valid transaction dates.

SELECT *
FROM gold.fact_sales
WHERE TxnDate IS NULL;